# 04 — Training Evaluation & Visualisation

Load artefacts produced by `src/train.py` and generate all diagnostic plots.

**Prerequisites** — run training first:
```bash
python -m src.train \
    --cohort cohort.csv --features features_hourly.csv \
    --output-dir checkpoints \
    --history-csv results/training_history.csv \
    --predictions-csv results/test_predictions.csv
```

Only **Cell 2** (path config) normally needs editing.

In [ ]:
# Cell 1 — Imports & path setup
import sys
from pathlib import Path

# Support running from both the project root and from inside the notebook dir
PROJECT_ROOT = Path(".").resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")   # ensure headless-safe backend
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

from src.viz import (
    plot_training_curves,
    plot_roc_curve,
    plot_pr_curve,
    plot_calibration,
    plot_score_distribution,
    plot_confusion_at_threshold,
    make_all_plots,
)
from src.train import bootstrap_ci

print("Imports OK.  PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
# Cell 2 — Path configuration (edit these if your paths differ)
CHECKPOINT    = PROJECT_ROOT / "checkpoints" / "best_model.pt"
HISTORY_CSV   = PROJECT_ROOT / "results" / "training_history.csv"
PREDS_CSV     = PROJECT_ROOT / "results" / "test_predictions.csv"
RESULTS_DIR   = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Checkpoint    : {CHECKPOINT}  exists={CHECKPOINT.exists()}")
print(f"History CSV   : {HISTORY_CSV}  exists={HISTORY_CSV.exists()}")
print(f"Predictions   : {PREDS_CSV}  exists={PREDS_CSV.exists()}")

In [ ]:
# Cell 3 — Training history summary
try:
    history = pd.read_csv(HISTORY_CSV)
    print(f"Training ran for {len(history)} epochs.")
    print("\nLast 10 epochs:")
    display(history.tail(10))
    best_idx  = history["val_auroc"].idxmax()
    best_row  = history.iloc[best_idx]
    print(f"\nBest val AUROC: {best_row['val_auroc']:.4f}  "
          f"(epoch {int(best_row['epoch'])}, "
          f"val AUPRC {best_row['val_auprc']:.4f})")
except FileNotFoundError:
    print(f"WARNING: {HISTORY_CSV} not found — run training first.")
    history = None

In [ ]:
# Cell 4 — Training curves plot
if history is not None:
    out = RESULTS_DIR / "training_curves.png"
    plot_training_curves(HISTORY_CSV, output_path=out, show=False)
    img = mpimg.imread(out)
    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(img)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    print(f"Saved → {out}")
else:
    print("Skipped — no history CSV.")

In [ ]:
# Cell 5 — Load test predictions & sanity checks
preds = pd.read_csv(PREDS_CSV)
labels = preds["label"].to_numpy(dtype=int)
probs  = preds["prob"].to_numpy(dtype=float)
prevalence = float(labels.mean())

assert not np.isnan(probs).any(), "NaN in predictions — check train.py evaluate()"
assert probs.min() >= 0 and probs.max() <= 1, "Probs outside [0, 1]"

print(f"Test set  : {len(preds):,} samples")
print(f"Positives : {labels.sum():,} ({100 * prevalence:.1f}%)")
print(f"Prob range: [{probs.min():.4f}, {probs.max():.4f}]")

In [ ]:
# Cell 6 — Checkpoint metadata
if CHECKPOINT.exists():
    ckpt = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
    print(f"Checkpoint epoch  : {ckpt['epoch']}")
    print(f"Val AUROC         : {ckpt['val_auroc']:.4f}")
    print(f"Val AUPRC         : {ckpt['val_auprc']:.4f}")
    print("\nTraining args:")
    for k, v in sorted(ckpt["args"].items()):
        print(f"  {k:<22} {v}")
else:
    print(f"WARNING: {CHECKPOINT} not found.")

In [ ]:
# Cell 7 — Bootstrap CI (200 iterations)
print("Computing bootstrap CIs (200 iterations) …")
ci = bootstrap_ci(labels, probs, n_iter=200, seed=42)

auroc_val = roc_auc_score(labels, probs)
auprc_val = average_precision_score(labels, probs)

print(f"AUROC : {auroc_val:.4f}  95% CI [{ci['auroc'][0]:.4f}, {ci['auroc'][1]:.4f}]")
print(f"AUPRC : {auprc_val:.4f}  95% CI [{ci['auprc'][0]:.4f}, {ci['auprc'][1]:.4f}]")
print(f"\nDeLLiriuM benchmark (structured-EHR deep learning): ~0.781 AUROC")
print(f"DeLLiriuM benchmark (LLM):                          ~0.825 AUROC")

In [ ]:
# Cell 8 — ROC curve with bootstrap CI band
out = RESULTS_DIR / "roc_curve.png"
plot_roc_curve(
    labels, probs,
    auroc_ci=ci["auroc"],
    bootstrap_n=200,
    output_path=out,
    show=False,
)
img = mpimg.imread(out)
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img); ax.axis("off")
plt.tight_layout(); plt.show()
print(f"Saved → {out}")

In [ ]:
# Cell 9 — Precision-Recall curve
out = RESULTS_DIR / "pr_curve.png"
plot_pr_curve(
    labels, probs,
    auprc_ci=ci["auprc"],
    prevalence=prevalence,
    output_path=out,
    show=False,
)
img = mpimg.imread(out)
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img); ax.axis("off")
plt.tight_layout(); plt.show()
print(f"Saved → {out}")

In [ ]:
# Cell 10 — Calibration (quantile bins)
out = RESULTS_DIR / "calibration.png"
plot_calibration(
    labels, probs,
    n_bins=10,
    strategy="quantile",
    output_path=out,
    show=False,
)
img = mpimg.imread(out)
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img); ax.axis("off")
plt.tight_layout(); plt.show()
print(f"Saved → {out}")

In [ ]:
# Cell 11 — Score distribution violin
out = RESULTS_DIR / "score_distribution.png"
plot_score_distribution(
    labels, probs,
    output_path=out,
    show=False,
)
img = mpimg.imread(out)
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img); ax.axis("off")
plt.tight_layout(); plt.show()
print(f"Saved → {out}")

In [ ]:
# Cell 12 — Confusion matrices at 0.5 and Youden's J optimal threshold
fpr, tpr, thresholds = roc_curve(labels, probs)
opt_thr = float(thresholds[np.argmax(tpr - fpr)])
print(f"Optimal threshold (Youden's J): {opt_thr:.4f}")

out_050 = RESULTS_DIR / "confusion_at_0.50.png"
m_050 = plot_confusion_at_threshold(
    labels, probs, threshold=0.5,
    output_path=out_050, show=False,
)
print(f"At threshold 0.50: {m_050}")

out_opt = RESULTS_DIR / f"confusion_optimal_{opt_thr:.3f}.png"
m_opt = plot_confusion_at_threshold(
    labels, probs, threshold=opt_thr,
    output_path=out_opt, show=False,
)
print(f"At optimal {opt_thr:.4f}: {m_opt}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, path, title in zip(axes, [out_050, out_opt],
                            ["Threshold = 0.50", f"Optimal = {opt_thr:.3f}"]):
    ax.imshow(mpimg.imread(path)); ax.axis("off"); ax.set_title(title)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 13 — Final metrics summary table
results_df = pd.DataFrame([{
    "AUROC":                round(auroc_val, 4),
    "AUROC 95% CI":         f"[{ci['auroc'][0]:.4f}, {ci['auroc'][1]:.4f}]",
    "AUPRC":                round(auprc_val, 4),
    "AUPRC 95% CI":         f"[{ci['auprc'][0]:.4f}, {ci['auprc'][1]:.4f}]",
    "Sensitivity@0.50":     round(m_050["sensitivity"], 4),
    "Specificity@0.50":     round(m_050["specificity"], 4),
    "F1@0.50":              round(m_050["f1"], 4),
    f"Sensitivity@{opt_thr:.3f}": round(m_opt["sensitivity"], 4),
    f"Specificity@{opt_thr:.3f}": round(m_opt["specificity"], 4),
    f"F1@{opt_thr:.3f}":          round(m_opt["f1"], 4),
    "N test":               len(labels),
    "Prevalence":           f"{100 * prevalence:.1f}%",
    "DeLLiriuM target (struct EHR)": "~0.781 AUROC",
}])

display(results_df.T.rename(columns={0: "Value"}))

metrics_out = RESULTS_DIR / "final_metrics.csv"
results_df.to_csv(metrics_out, index=False)
print(f"\nFinal metrics saved → {metrics_out}")